# 05 - Physics Moment Terms

This notebook isolates the moment-based physics losses that operate on the
reconstructed profile treated as a (non-normalised) distribution over
elevation: `total_power` (zeroth moment / mass) and `moments` (a weighted
combination of mass, mean, and spread mismatches). We perturb the
prediction's mass, mean, and spread one at a time and confirm each term
responds to the corresponding perturbation.

Modules exercised: `PhysicsComponents.total_power`, `PhysicsComponents.moments`,
`PhysicsComponents.moment_sums`.

In [ ]:
import sys
from pathlib import Path

REPO_ROOT = Path("../..").resolve()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import numpy             as np
import torch
import matplotlib.pyplot as plt

SEED   = 0
DEVICE = torch.device("cpu")

np.random.seed(SEED)
torch.manual_seed(SEED)
torch.use_deterministic_algorithms(True, warn_only=True)

plt.rcParams.update({
    "figure.dpi"     : 120,
    "savefig.dpi"    : 300,
    "font.size"      : 10,
    "axes.grid"      : True,
    "grid.alpha"     : 0.3,
    "axes.spines.top"   : False,
    "axes.spines.right" : False,
})

print("repo root :", REPO_ROOT)
print("torch     :", torch.__version__)
print("device    :", DEVICE)


In [ ]:
from configuration.training_config import GaussianConfig, GeometryConfig, LossConfig
from pipelines.backbone_pipeline.loss import Loss, PhysicsComponents as PC
from tools import NullLogger, NullTracker

class IdentityNormStats:
    def normalize_output(self, tensor):
        return tensor

    def denormalize_output(self, tensor):
        return tensor

X_MIN, X_MAX = -10.0, 40.0
x_axis       = torch.linspace(X_MIN, X_MAX, 96, dtype=torch.float32)
gaussian_cfg = GaussianConfig(n_default_gaussians=1, x_min=X_MIN, x_max=X_MAX)
criterion    = Loss(x_axis, NullLogger(), NullTracker(), gaussian_cfg,
                    LossConfig(), IdentityNormStats(), GeometryConfig())
dx           = criterion.dx
floor        = 1e-3

def curve(amp, mu, sig, H=3, W=3):
    p = torch.tensor([amp, mu, sig], dtype=torch.float32).reshape(1, 3, 1, 1)
    return criterion.reconstruct_gaussians(p.expand(1, 3, H, W).contiguous())

target = curve(2.0, 12.0, 5.0)
s0, s1, s2 = PC.moment_sums(target, x_axis, dx)
print('target mass  :', round(float(s0.mean()), 4))
print('target mean  :', round(float((s1 / s0).mean()), 4))


## Total power vs mass mismatch

Scaling the predicted amplitude changes the integrated power. `total_power`
measures the relative mass error and should be zero at scale 1 and grow as
the prediction over- or under-shoots the target mass.

In [ ]:
scales = torch.linspace(0.3, 2.0, 35)
tp = [float(PC.total_power(curve(2.0 * float(k), 12.0, 5.0), target, dx, floor)) for k in scales]

fig, ax = plt.subplots(figsize=(6.5, 4))
ax.plot(scales.numpy(), tp, color='C0')
ax.axvline(1.0, color='grey', ls=':', lw=1, label='matched mass')
ax.set_xlabel('predicted amplitude scale')
ax.set_ylabel('total_power loss (relative)')
ax.set_title('Total power vs mass mismatch')
ax.legend(frameon=False)
fig.tight_layout()
plt.show()


## Moment term decomposed by perturbation type

The `moments` term combines mass, mean, and spread mismatches. We probe it
three ways: amplitude scaling (mass), centre shift (mean), and width change
(spread). Each curve should bottom out at the unperturbed value.

In [ ]:
moments_w = (1.0, 1.0, 1.0)

amp_k   = torch.linspace(0.3, 2.0, 35)
mean_d  = torch.linspace(-12.0, 12.0, 49)
spread_k= torch.linspace(0.4, 2.5, 35)

m_mass   = [float(PC.moments(curve(2.0 * float(k), 12.0, 5.0), target, x_axis, dx, floor, moments_w)) for k in amp_k]
m_mean   = [float(PC.moments(curve(2.0, 12.0 + float(d), 5.0), target, x_axis, dx, floor, moments_w)) for d in mean_d]
m_spread = [float(PC.moments(curve(2.0, 12.0, 5.0 * float(k)), target, x_axis, dx, floor, moments_w)) for k in spread_k]

fig, axes = plt.subplots(1, 3, figsize=(12, 3.4))
axes[0].plot(amp_k.numpy(), m_mass);    axes[0].axvline(1.0, color='grey', ls=':', lw=1)
axes[0].set_title('mass (amplitude scale)'); axes[0].set_xlabel('scale')
axes[1].plot(mean_d.numpy(), m_mean);    axes[1].axvline(0.0, color='grey', ls=':', lw=1)
axes[1].set_title('mean (centre shift)');   axes[1].set_xlabel('shift [m]')
axes[2].plot(spread_k.numpy(), m_spread);axes[2].axvline(1.0, color='grey', ls=':', lw=1)
axes[2].set_title('spread (width scale)');  axes[2].set_xlabel('scale')
for ax in axes:
    ax.set_ylabel('moments loss')
fig.tight_layout()
plt.show()


## Expected visual outcome

`total_power` is a V-shaped relative error with its minimum at amplitude
scale 1. Each panel of the `moments` decomposition reaches its minimum at
the unperturbed setting (scale 1, shift 0, scale 1) and increases away from
it, confirming that the combined moment term responds to mass, mean, and
spread errors respectively.